# Four Qwen3-VL-8B experiments — separate runs and stages

Use Python 3.11 / Python (afterimage_env). Start with Configure paths and Shared setup. The default route is Shared preflight, then each run's separate Training, Verify adapter, and Evaluation cells. Each run also has independent saved-configuration, status/results, and log-view cells. You can return to one failed stage without launching another run.

For the original unattended route, set UNATTENDED_QUEUE = True and run the short Optional unattended queue cell after setup. Individual stage cells then skip execution; all the per-run inspection cells still work. Keep the kernel/computer awake while a worker is active. Interrupting a launch cell stops and waits for its active child.

Opening this notebook or executing configuration/setup starts no audit, preprocessing, training, evaluation, or model loading. All saved outputs are empty. ORIGINAL_IMAGE_INDEX is needed for verified CLAHE ablation; see README.md.


## Configure paths


In [1]:
# Configure paths — definitions only; no audit, preprocessing, imports of model code, or launch.
from pathlib import Path

PROJECT_ROOT = Path(r"C:\Users\moht2\Documents\Afterimage")
REPRODUCTION = PROJECT_ROOT / "finetuning/Qwen3/artifacts/paper_reproduction"
REVIEWED_CONFIGURATION = REPRODUCTION / "training_batches/aptos_paper_same_seed_three_runs_001/paper_training_configuration.json"
DATASET_MANIFEST = REPRODUCTION / "data/4fda605471d5430c3733/dataset_manifest.json"
TARGET_ADAPTER_CONFIGURATION = REVIEWED_CONFIGURATION.parent / "run_01/best_lora/adapter_config.json"
IMAGE_ROOT = Path(r"C:\Users\moht2\Downloads\DR_Aptos_Messidor_Only_Full_Dataset\Res_512")
CLAHE_ON_IMAGES = IMAGE_ROOT / "Aptos_messidor_external_CLAHE"
CLAHE_OFF_IMAGES = IMAGE_ROOT / "Aptos_messidor_external"

# CSV of independent originals, needed to verify the CLAHE ablation.
# Columns: source_dataset,image_id,original_path,is_original
ORIGINAL_IMAGE_INDEX = None  # e.g. Path(r"D:\DR evidence\original_image_index.csv")
BASE_SNAPSHOT = None  # None uses the exact cached snapshot in REVIEWED_CONFIGURATION.

# Keep this directory to restart this queue; use a new name for a fresh experiment.
BATCH_DIR = PROJECT_ROOT / "experiments/artifacts/qwen_four_run_001"
WORKER = PROJECT_ROOT / "experiments/qwen_experiment_worker.py"
GPU_INDEX = "0"
MINIMUM_DISK_GIB = 80.0

# Default: independent per-run stage cells. True enables the optional unattended queue.
UNATTENDED_QUEUE = False

# Run only ranks 16, 8 and 32 with CLAHE on; True also requests the CLAHE-off ablation.
INCLUDE_CLAHE_OFF = False


## Shared setup — definitions only

This loads the standard-library notebook controller. Scientific checks, images and models remain inside explicitly launched workers. The launch/cleanup implementation is NotebookOrchestrator in qwen_experiment_worker.py; the cells below call named stages.


In [2]:
import importlib.util
import sys

sys.dont_write_bytecode = True
worker_path = WORKER.expanduser().resolve()
if not worker_path.is_file():
    raise FileNotFoundError(worker_path)
spec = importlib.util.spec_from_file_location("qwen_notebook_control", worker_path)
control_module = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = control_module
spec.loader.exec_module(control_module)

request = {
    "project_root": str(PROJECT_ROOT.expanduser().resolve()),
    "reviewed_configuration": str(REVIEWED_CONFIGURATION.expanduser().resolve()),
    "dataset_manifest": str(DATASET_MANIFEST.expanduser().resolve()),
    "target_adapter_configuration": str(TARGET_ADAPTER_CONFIGURATION.expanduser().resolve()),
    "clahe_on_images": str(CLAHE_ON_IMAGES.expanduser().resolve()),
    "clahe_off_images": str(CLAHE_OFF_IMAGES.expanduser().resolve()),
    "original_image_index": str(ORIGINAL_IMAGE_INDEX.expanduser().resolve()) if ORIGINAL_IMAGE_INDEX else None,
    "base_snapshot": str(BASE_SNAPSHOT.expanduser().resolve()) if BASE_SNAPSHOT else None,
    "minimum_disk_gib": MINIMUM_DISK_GIB,
    "include_clahe_off": INCLUDE_CLAHE_OFF,
}
orchestrator = control_module.NotebookOrchestrator(request, BATCH_DIR, gpu_index=GPU_INDEX)
print("Shared setup ready. No workers have been launched.")


Shared setup ready. No workers have been launched.


## Shared preflight

Run this before individual stages. It checks the environment, frozen cohorts, image branches, base snapshot and disk space, then exits. It starts no training. A shared blocker stops subsequent stage launches. An unverifiable off branch blocks only the CLAHE ablation.

The next two cells read saved preflight results/logs without rerunning it.


In [3]:
if not UNATTENDED_QUEUE:
    orchestrator.preflight()
else:
    print("Unattended mode: the optional queue cell will run preflight.")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
W1110 08:53:06.535000 8444 site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1110 08:53:06.627000 8444 site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
Unsloth: xformers is installed but cannot load with torch 2.14.1+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything 

In [4]:
# Read saved preflight results only.
orchestrator.show_preflight()


{
  "started_at": "2026-11-10T05:52:41+00:00",
  "status": "ready",
  "shared_blockers": [],
  "ablation_blockers": [],
  "checks": [
    "Frozen counts, class counts, unique IDs, recovered labels/membership, mapping and image/patient split overlap passed; Messidor is external only",
    "No image augmentation in the new formatting/preprocessing paths; pinned collator source performs deterministic resizing/formatting, and weighted sampling applies to train only",
    "Shared CLAHE branch: exact 4,719 filenames, valid RGB 512x512 PNGs, byte fingerprints; no generated augmentation variants accepted",
    "CLAHE-off explicitly excluded; off-branch pairing and original-image-index checks skipped"
  ],
  "excluded_runs": [
    "r16_clahe_off"
  ],
  "resources": {
    "disk_free_gib": 196.64960098266602,
    "gpu": "NVIDIA GeForce RTX 5070 Ti",
    "gpu_identity": {
      "name": "NVIDIA GeForce RTX 5070 Ti",
      "total_memory": 17066360832,
      "capability": [
        12,
        0
   

In [5]:
# Read the end of the saved preflight log only.
orchestrator.show_log(None, "preflight", lines=40)


Log: C:\Users\moht2\Documents\Afterimage\experiments\artifacts\qwen_four_run_001\preflight.log

=== 2026-11-10T05:52:41+00:00 ===
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
W1110 08:53:06.535000 8444 site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1110 08:53:06.627000 8444 site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
Unsloth: xformers is installed but cannot load with torch 2.14.1+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`,

## Optional unattended queue

Use this route only when UNATTENDED_QUEUE = True. This short launcher performs preflight and queues all eligible runs in the fixed order. Their saved artifacts remain accessible in the separate sections below. With the default False, this cell starts nothing.

The queue preserves the same fixed settings, validation-based stopping rule and roughly 12-hour planning window. Isolated failures continue to the next eligible run; a shared failure or interruption ends the queue.


In [6]:
if UNATTENDED_QUEUE:
    orchestrator.run_all()
else:
    print("Independent-stage mode: use the separate run cells below.")


Independent-stage mode: use the separate run cells below.


## Run 1 — r16_clahe_on

Rank 16; CLAHE On; alpha/r 1. Fresh shared control for both comparisons.

Run Training, Verify adapter and Evaluation separately. Evaluation also rechecks the retained adapter in a CPU child before loading its model. Configuration/results/log cells only read saved files. In unattended mode the three action cells skip; the inspection cells remain available.


In [7]:
# Saved resolved configuration — r16_clahe_on
orchestrator.show_configuration("r16_clahe_on")


{
  "version": 1,
  "shared_identity": "5b933cda14d22e5b7fd302805fcd908c18992e71ada8c5fa8af77b28c6747149",
  "run_id": "r16_clahe_on",
  "rank": 16,
  "clahe": true,
  "selected": true,
  "eligible": true,
  "exclusion_reason": "",
  "training": {
    "epochs": 50,
    "batch_size": 4,
    "grad_accum": 2,
    "lr": 0.0005,
    "warmup_steps": 20,
    "max_length": 2048,
    "weight_decay": 0.001,
    "optim": "adamw_8bit",
    "scheduler": "linear",
    "lora_r": 16,
    "lora_alpha": 16,
    "lora_dropout": 0.05,
    "bias": "none",
    "gradient_checkpointing": true,
    "img_size": 512,
    "load_in_4bit": true,
    "eval_strategy": "epoch",
    "save_strategy": "epoch",
    "save_total_limit": 2,
    "early_stopping_patience": 2,
    "early_stopping_threshold": 0.0,
    "metric_for_best_model": "eval_loss",
    "logging_steps": 5,
    "trainer_seed": 42,
    "eval_batch_size": 8,
    "lora_seed_mode": "original_unsloth_default",
    "expected_lora_default_seed": 3407
  },
  "base_

In [8]:
# Training — r16_clahe_on
if not UNATTENDED_QUEUE:
    orchestrator.run_stage("train", "r16_clahe_on")


[2026-11-10T05:54:04+00:00] Rechecking source, configuration, frozen CSV/image and cached-base hashes: r16_clahe_on
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
W1110 08:54:16.964000 32580 site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1110 08:54:17.018000 32580 site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
Unsloth: xformers is installed but cannot load with torch 2.14.1+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uni

In [9]:
# Verify saved adapter — r16_clahe_on
if not UNATTENDED_QUEUE:
    orchestrator.run_stage("verify", "r16_clahe_on")


[2026-11-10T07:18:39+00:00] Rechecking source, configuration, frozen CSV/image and cached-base hashes: r16_clahe_on
[2026-11-10T07:18:47+00:00] Saved adapter verified after training process exit: r16_clahe_on
[2026-11-10T07:18:47+00:00] Batch summary: C:\Users\moht2\Documents\Afterimage\experiments\artifacts\qwen_four_run_001\batch_summary.csv; report: C:\Users\moht2\Documents\Afterimage\experiments\artifacts\qwen_four_run_001\batch_report.md
[2026-11-10T07:18:47+00:00] r16_clahe_on / verify: exit 0; log=C:\Users\moht2\Documents\Afterimage\experiments\artifacts\qwen_four_run_001\r16_clahe_on\verify.log
[2026-11-10T07:18:47+00:00] Batch summary: C:\Users\moht2\Documents\Afterimage\experiments\artifacts\qwen_four_run_001\batch_summary.csv; report: C:\Users\moht2\Documents\Afterimage\experiments\artifacts\qwen_four_run_001\batch_report.md
[2026-11-10T07:18:47+00:00] batch / summarize: exit 0; log=C:\Users\moht2\Documents\Afterimage\experiments\artifacts\qwen_four_run_001\summarize.log
[20

In [10]:
# Evaluation on both frozen cohorts — r16_clahe_on
if not UNATTENDED_QUEUE:
    orchestrator.run_stage("evaluate", "r16_clahe_on")


[2026-11-10T07:18:47+00:00] Rechecking source, configuration, frozen CSV/image and cached-base hashes: r16_clahe_on
[2026-11-10T07:18:54+00:00] Saved adapter verified after training process exit: r16_clahe_on
[2026-11-10T07:18:54+00:00] Batch summary: C:\Users\moht2\Documents\Afterimage\experiments\artifacts\qwen_four_run_001\batch_summary.csv; report: C:\Users\moht2\Documents\Afterimage\experiments\artifacts\qwen_four_run_001\batch_report.md
[2026-11-10T07:18:54+00:00] r16_clahe_on / verify: exit 0; log=C:\Users\moht2\Documents\Afterimage\experiments\artifacts\qwen_four_run_001\r16_clahe_on\verify.log
[2026-11-10T07:18:55+00:00] Rechecking source, configuration, frozen CSV/image and cached-base hashes: r16_clahe_on
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
W1110 10:19:07.938000 22788 site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling regis

In [11]:
# Saved stage status, adapter settings and metrics — r16_clahe_on
orchestrator.show_run("r16_clahe_on")


Run: r16_clahe_on
Artifacts: C:\Users\moht2\Documents\Afterimage\experiments\artifacts\qwen_four_run_001\r16_clahe_on
Selected: True; eligible: True; exclusion: ; blockers: []

train: {
  "attempts": [
    {
      "started_at": "2026-11-10T05:54:04+00:00",
      "status": "complete",
      "ended_at": "2026-11-10T07:18:34+00:00",
      "elapsed_seconds": 5069.562
    }
  ],
  "status": "complete",
  "updated_at": "2026-11-10T07:18:34+00:00",
  "elapsed_seconds": 5069.562
}

verify: {
  "attempts": [
    {
      "started_at": "2026-11-10T07:18:39+00:00",
      "status": "complete",
      "ended_at": "2026-11-10T07:18:47+00:00",
      "elapsed_seconds": 7.860000000000582
    },
    {
      "started_at": "2026-11-10T07:18:47+00:00",
      "status": "complete",
      "ended_at": "2026-11-10T07:18:54+00:00",
      "elapsed_seconds": 6.95299999999952
    }
  ],
  "status": "complete",
  "updated_at": "2026-11-10T07:18:54+00:00",
  "elapsed_seconds": 6.95299999999952
}

evaluate: {
  "attempt

In [12]:
# Saved train log — r16_clahe_on; increase lines to inspect more history.
orchestrator.show_log("r16_clahe_on", "train", lines=40)


Log: C:\Users\moht2\Documents\Afterimage\experiments\artifacts\qwen_four_run_001\r16_clahe_on\train.log
 85%|████████▍ | 39/46 [00:28<00:05,  1.34it/s]

 87%|████████▋ | 40/46 [00:28<00:04,  1.34it/s]

 89%|████████▉ | 41/46 [00:29<00:03,  1.34it/s]

 91%|█████████▏| 42/46 [00:30<00:02,  1.34it/s]

 93%|█████████▎| 43/46 [00:31<00:02,  1.34it/s]

 96%|█████████▌| 44/46 [00:31<00:01,  1.34it/s]

 98%|█████████▊| 45/46 [00:32<00:00,  1.35it/s]

100%|██████████| 46/46 [00:33<00:00,  1.48it/s]
                                                        


                                               
{'eval_loss': '0.01145', 'eval_runtime': '34.01', 'eval_samples_per_second': '10.76', 'eval_steps_per_second': '1.353', 'epoch': '5'}

 10%|█         | 1835/18350 [1:23:44<8:40:53,  1.89s/it]

100%|██████████| 46/46 [00:33<00:00,  1.48it/s]

                                               C:\Users\moht2\.conda\envs\afterimage_env\Lib\site-packages\peft\utils\save_and_load.py:313: UserWarning: Cou

In [13]:
# Saved verify log — r16_clahe_on; increase lines to inspect more history.
orchestrator.show_log("r16_clahe_on", "verify", lines=40)


Log: C:\Users\moht2\Documents\Afterimage\experiments\artifacts\qwen_four_run_001\r16_clahe_on\verify.log

=== 2026-11-10T07:18:39+00:00 ===
[2026-11-10T07:18:39+00:00] Rechecking source, configuration, frozen CSV/image and cached-base hashes: r16_clahe_on
[2026-11-10T07:18:47+00:00] Saved adapter verified after training process exit: r16_clahe_on
[2026-11-10T07:18:47+00:00] Batch summary: C:\Users\moht2\Documents\Afterimage\experiments\artifacts\qwen_four_run_001\batch_summary.csv; report: C:\Users\moht2\Documents\Afterimage\experiments\artifacts\qwen_four_run_001\batch_report.md

=== 2026-11-10T07:18:47+00:00 ===
[2026-11-10T07:18:47+00:00] Rechecking source, configuration, frozen CSV/image and cached-base hashes: r16_clahe_on
[2026-11-10T07:18:54+00:00] Saved adapter verified after training process exit: r16_clahe_on
[2026-11-10T07:18:54+00:00] Batch summary: C:\Users\moht2\Documents\Afterimage\experiments\artifacts\qwen_four_run_001\batch_summary.csv; report: C:\Users\moht2\Document

In [14]:
# Saved evaluate log — r16_clahe_on; increase lines to inspect more history.
orchestrator.show_log("r16_clahe_on", "evaluate", lines=40)


Log: C:\Users\moht2\Documents\Afterimage\experiments\artifacts\qwen_four_run_001\r16_clahe_on\evaluate.log
[2026-11-10T07:34:24+00:00] r16_clahe_on messidor_external: 690/1057
[2026-11-10T07:34:33+00:00] r16_clahe_on messidor_external: 700/1057
[2026-11-10T07:34:41+00:00] r16_clahe_on messidor_external: 710/1057
[2026-11-10T07:34:49+00:00] r16_clahe_on messidor_external: 720/1057
[2026-11-10T07:34:58+00:00] r16_clahe_on messidor_external: 730/1057
[2026-11-10T07:35:06+00:00] r16_clahe_on messidor_external: 740/1057
[2026-11-10T07:35:14+00:00] r16_clahe_on messidor_external: 750/1057
[2026-11-10T07:35:23+00:00] r16_clahe_on messidor_external: 760/1057
[2026-11-10T07:35:31+00:00] r16_clahe_on messidor_external: 770/1057
[2026-11-10T07:35:39+00:00] r16_clahe_on messidor_external: 780/1057
[2026-11-10T07:35:48+00:00] r16_clahe_on messidor_external: 790/1057
[2026-11-10T07:35:56+00:00] r16_clahe_on messidor_external: 800/1057
[2026-11-10T07:36:04+00:00] r16_clahe_on messidor_external: 810/1

## Run 2 — r8_clahe_on

Rank 8; CLAHE On; alpha/r 2. Rank comparison against the fresh rank-16 CLAHE-on control.

Run Training, Verify adapter and Evaluation separately. Evaluation also rechecks the retained adapter in a CPU child before loading its model. Configuration/results/log cells only read saved files. In unattended mode the three action cells skip; the inspection cells remain available.


In [15]:
# Saved resolved configuration — r8_clahe_on
orchestrator.show_configuration("r8_clahe_on")


{
  "version": 1,
  "shared_identity": "5b933cda14d22e5b7fd302805fcd908c18992e71ada8c5fa8af77b28c6747149",
  "run_id": "r8_clahe_on",
  "rank": 8,
  "clahe": true,
  "selected": true,
  "eligible": true,
  "exclusion_reason": "",
  "training": {
    "epochs": 50,
    "batch_size": 4,
    "grad_accum": 2,
    "lr": 0.0005,
    "warmup_steps": 20,
    "max_length": 2048,
    "weight_decay": 0.001,
    "optim": "adamw_8bit",
    "scheduler": "linear",
    "lora_r": 8,
    "lora_alpha": 16,
    "lora_dropout": 0.05,
    "bias": "none",
    "gradient_checkpointing": true,
    "img_size": 512,
    "load_in_4bit": true,
    "eval_strategy": "epoch",
    "save_strategy": "epoch",
    "save_total_limit": 2,
    "early_stopping_patience": 2,
    "early_stopping_threshold": 0.0,
    "metric_for_best_model": "eval_loss",
    "logging_steps": 5,
    "trainer_seed": 42,
    "eval_batch_size": 8,
    "lora_seed_mode": "original_unsloth_default",
    "expected_lora_default_seed": 3407
  },
  "base_rev

In [16]:
# Training — r8_clahe_on
if not UNATTENDED_QUEUE:
    orchestrator.run_stage("train", "r8_clahe_on")


[2026-11-10T07:39:32+00:00] Rechecking source, configuration, frozen CSV/image and cached-base hashes: r8_clahe_on
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
W1110 10:39:44.828000 31468 site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1110 10:39:44.883000 31468 site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
Unsloth: xformers is installed but cannot load with torch 2.14.1+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip unin

In [17]:
# Verify saved adapter — r8_clahe_on
if not UNATTENDED_QUEUE:
    orchestrator.run_stage("verify", "r8_clahe_on")


[2026-11-10T09:03:53+00:00] Rechecking source, configuration, frozen CSV/image and cached-base hashes: r8_clahe_on
[2026-11-10T09:04:01+00:00] Saved adapter verified after training process exit: r8_clahe_on
[2026-11-10T09:04:01+00:00] Batch summary: C:\Users\moht2\Documents\Afterimage\experiments\artifacts\qwen_four_run_001\batch_summary.csv; report: C:\Users\moht2\Documents\Afterimage\experiments\artifacts\qwen_four_run_001\batch_report.md
[2026-11-10T09:04:01+00:00] r8_clahe_on / verify: exit 0; log=C:\Users\moht2\Documents\Afterimage\experiments\artifacts\qwen_four_run_001\r8_clahe_on\verify.log
[2026-11-10T09:04:01+00:00] Batch summary: C:\Users\moht2\Documents\Afterimage\experiments\artifacts\qwen_four_run_001\batch_summary.csv; report: C:\Users\moht2\Documents\Afterimage\experiments\artifacts\qwen_four_run_001\batch_report.md
[2026-11-10T09:04:01+00:00] batch / summarize: exit 0; log=C:\Users\moht2\Documents\Afterimage\experiments\artifacts\qwen_four_run_001\summarize.log
[2026-1

In [18]:
# Evaluation on both frozen cohorts — r8_clahe_on
if not UNATTENDED_QUEUE:
    orchestrator.run_stage("evaluate", "r8_clahe_on")


[2026-11-10T09:04:02+00:00] Rechecking source, configuration, frozen CSV/image and cached-base hashes: r8_clahe_on
[2026-11-10T09:04:08+00:00] Saved adapter verified after training process exit: r8_clahe_on
[2026-11-10T09:04:09+00:00] Batch summary: C:\Users\moht2\Documents\Afterimage\experiments\artifacts\qwen_four_run_001\batch_summary.csv; report: C:\Users\moht2\Documents\Afterimage\experiments\artifacts\qwen_four_run_001\batch_report.md
[2026-11-10T09:04:09+00:00] r8_clahe_on / verify: exit 0; log=C:\Users\moht2\Documents\Afterimage\experiments\artifacts\qwen_four_run_001\r8_clahe_on\verify.log
[2026-11-10T09:04:09+00:00] Rechecking source, configuration, frozen CSV/image and cached-base hashes: r8_clahe_on
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
W1110 12:04:35.187000 1824 site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_co

In [19]:
# Saved stage status, adapter settings and metrics — r8_clahe_on
orchestrator.show_run("r8_clahe_on")


Run: r8_clahe_on
Artifacts: C:\Users\moht2\Documents\Afterimage\experiments\artifacts\qwen_four_run_001\r8_clahe_on
Selected: True; eligible: True; exclusion: ; blockers: []

train: {
  "attempts": [
    {
      "started_at": "2026-11-10T07:39:32+00:00",
      "status": "complete",
      "ended_at": "2026-11-10T09:03:49+00:00",
      "elapsed_seconds": 5056.516
    }
  ],
  "status": "complete",
  "updated_at": "2026-11-10T09:03:49+00:00",
  "elapsed_seconds": 5056.516
}

verify: {
  "attempts": [
    {
      "started_at": "2026-11-10T09:03:53+00:00",
      "status": "complete",
      "ended_at": "2026-11-10T09:04:01+00:00",
      "elapsed_seconds": 7.45299999999952
    },
    {
      "started_at": "2026-11-10T09:04:02+00:00",
      "status": "complete",
      "ended_at": "2026-11-10T09:04:08+00:00",
      "elapsed_seconds": 6.828000000001339
    }
  ],
  "status": "complete",
  "updated_at": "2026-11-10T09:04:08+00:00",
  "elapsed_seconds": 6.828000000001339
}

evaluate: {
  "attempts

In [20]:
# Saved train log — r8_clahe_on; increase lines to inspect more history.
orchestrator.show_log("r8_clahe_on", "train", lines=40)


Log: C:\Users\moht2\Documents\Afterimage\experiments\artifacts\qwen_four_run_001\r8_clahe_on\train.log
 85%|████████▍ | 39/46 [00:27<00:05,  1.36it/s]

 87%|████████▋ | 40/46 [00:28<00:04,  1.36it/s]

 89%|████████▉ | 41/46 [00:29<00:03,  1.36it/s]

 91%|█████████▏| 42/46 [00:30<00:02,  1.37it/s]

 93%|█████████▎| 43/46 [00:30<00:02,  1.37it/s]

 96%|█████████▌| 44/46 [00:31<00:01,  1.37it/s]

 98%|█████████▊| 45/46 [00:32<00:00,  1.37it/s]

100%|██████████| 46/46 [00:32<00:00,  1.51it/s]
                                                        


                                               
{'eval_loss': '0.01903', 'eval_runtime': '33.59', 'eval_samples_per_second': '10.89', 'eval_steps_per_second': '1.369', 'epoch': '5'}

 10%|█         | 1835/18350 [1:23:38<8:37:03,  1.88s/it]

100%|██████████| 46/46 [00:32<00:00,  1.51it/s]

                                               C:\Users\moht2\.conda\envs\afterimage_env\Lib\site-packages\peft\utils\save_and_load.py:313: UserWarning: Coul

In [21]:
# Saved verify log — r8_clahe_on; increase lines to inspect more history.
orchestrator.show_log("r8_clahe_on", "verify", lines=40)


Log: C:\Users\moht2\Documents\Afterimage\experiments\artifacts\qwen_four_run_001\r8_clahe_on\verify.log

=== 2026-11-10T09:03:53+00:00 ===
[2026-11-10T09:03:53+00:00] Rechecking source, configuration, frozen CSV/image and cached-base hashes: r8_clahe_on
[2026-11-10T09:04:01+00:00] Saved adapter verified after training process exit: r8_clahe_on
[2026-11-10T09:04:01+00:00] Batch summary: C:\Users\moht2\Documents\Afterimage\experiments\artifacts\qwen_four_run_001\batch_summary.csv; report: C:\Users\moht2\Documents\Afterimage\experiments\artifacts\qwen_four_run_001\batch_report.md

=== 2026-11-10T09:04:01+00:00 ===
[2026-11-10T09:04:02+00:00] Rechecking source, configuration, frozen CSV/image and cached-base hashes: r8_clahe_on
[2026-11-10T09:04:08+00:00] Saved adapter verified after training process exit: r8_clahe_on
[2026-11-10T09:04:09+00:00] Batch summary: C:\Users\moht2\Documents\Afterimage\experiments\artifacts\qwen_four_run_001\batch_summary.csv; report: C:\Users\moht2\Documents\Aft

In [22]:
# Saved evaluate log — r8_clahe_on; increase lines to inspect more history.
orchestrator.show_log("r8_clahe_on", "evaluate", lines=40)


Log: C:\Users\moht2\Documents\Afterimage\experiments\artifacts\qwen_four_run_001\r8_clahe_on\evaluate.log
[2026-11-10T09:19:48+00:00] r8_clahe_on messidor_external: 690/1057
[2026-11-10T09:19:56+00:00] r8_clahe_on messidor_external: 700/1057
[2026-11-10T09:20:05+00:00] r8_clahe_on messidor_external: 710/1057
[2026-11-10T09:20:13+00:00] r8_clahe_on messidor_external: 720/1057
[2026-11-10T09:20:21+00:00] r8_clahe_on messidor_external: 730/1057
[2026-11-10T09:20:30+00:00] r8_clahe_on messidor_external: 740/1057
[2026-11-10T09:20:38+00:00] r8_clahe_on messidor_external: 750/1057
[2026-11-10T09:20:46+00:00] r8_clahe_on messidor_external: 760/1057
[2026-11-10T09:20:54+00:00] r8_clahe_on messidor_external: 770/1057
[2026-11-10T09:21:03+00:00] r8_clahe_on messidor_external: 780/1057
[2026-11-10T09:21:11+00:00] r8_clahe_on messidor_external: 790/1057
[2026-11-10T09:21:19+00:00] r8_clahe_on messidor_external: 800/1057
[2026-11-10T09:21:27+00:00] r8_clahe_on messidor_external: 810/1057
[2026-11-1

## Run 3 — r32_clahe_on

Rank 32; CLAHE On; alpha/r 0.5. Rank comparison against the same fresh rank-16 CLAHE-on control.

Run Training, Verify adapter and Evaluation separately. Evaluation also rechecks the retained adapter in a CPU child before loading its model. Configuration/results/log cells only read saved files. In unattended mode the three action cells skip; the inspection cells remain available.


In [23]:
# Saved resolved configuration — r32_clahe_on
orchestrator.show_configuration("r32_clahe_on")


{
  "version": 1,
  "shared_identity": "5b933cda14d22e5b7fd302805fcd908c18992e71ada8c5fa8af77b28c6747149",
  "run_id": "r32_clahe_on",
  "rank": 32,
  "clahe": true,
  "selected": true,
  "eligible": true,
  "exclusion_reason": "",
  "training": {
    "epochs": 50,
    "batch_size": 4,
    "grad_accum": 2,
    "lr": 0.0005,
    "warmup_steps": 20,
    "max_length": 2048,
    "weight_decay": 0.001,
    "optim": "adamw_8bit",
    "scheduler": "linear",
    "lora_r": 32,
    "lora_alpha": 16,
    "lora_dropout": 0.05,
    "bias": "none",
    "gradient_checkpointing": true,
    "img_size": 512,
    "load_in_4bit": true,
    "eval_strategy": "epoch",
    "save_strategy": "epoch",
    "save_total_limit": 2,
    "early_stopping_patience": 2,
    "early_stopping_threshold": 0.0,
    "metric_for_best_model": "eval_loss",
    "logging_steps": 5,
    "trainer_seed": 42,
    "eval_batch_size": 8,
    "lora_seed_mode": "original_unsloth_default",
    "expected_lora_default_seed": 3407
  },
  "base_

In [24]:
# Training — r32_clahe_on
if not UNATTENDED_QUEUE:
    orchestrator.run_stage("train", "r32_clahe_on")


[2026-11-10T09:24:55+00:00] Rechecking source, configuration, frozen CSV/image and cached-base hashes: r32_clahe_on
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
W1110 12:25:07.270000 10816 site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1110 12:25:07.326000 10816 site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
Unsloth: xformers is installed but cannot load with torch 2.14.1+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uni

In [25]:
# Verify saved adapter — r32_clahe_on
if not UNATTENDED_QUEUE:
    orchestrator.run_stage("verify", "r32_clahe_on")


[2026-11-10T10:18:12+00:00] Rechecking source, configuration, frozen CSV/image and cached-base hashes: r32_clahe_on
[2026-11-10T10:18:20+00:00] Saved adapter verified after training process exit: r32_clahe_on
[2026-11-10T10:18:20+00:00] Batch summary: C:\Users\moht2\Documents\Afterimage\experiments\artifacts\qwen_four_run_001\batch_summary.csv; report: C:\Users\moht2\Documents\Afterimage\experiments\artifacts\qwen_four_run_001\batch_report.md
[2026-11-10T10:18:20+00:00] r32_clahe_on / verify: exit 0; log=C:\Users\moht2\Documents\Afterimage\experiments\artifacts\qwen_four_run_001\r32_clahe_on\verify.log
[2026-11-10T10:18:21+00:00] Batch summary: C:\Users\moht2\Documents\Afterimage\experiments\artifacts\qwen_four_run_001\batch_summary.csv; report: C:\Users\moht2\Documents\Afterimage\experiments\artifacts\qwen_four_run_001\batch_report.md
[2026-11-10T10:18:21+00:00] batch / summarize: exit 0; log=C:\Users\moht2\Documents\Afterimage\experiments\artifacts\qwen_four_run_001\summarize.log
[20

In [26]:
# Evaluation on both frozen cohorts — r32_clahe_on
if not UNATTENDED_QUEUE:
    orchestrator.run_stage("evaluate", "r32_clahe_on")


[2026-11-10T10:18:21+00:00] Rechecking source, configuration, frozen CSV/image and cached-base hashes: r32_clahe_on
[2026-11-10T10:18:28+00:00] Saved adapter verified after training process exit: r32_clahe_on
[2026-11-10T10:18:29+00:00] Batch summary: C:\Users\moht2\Documents\Afterimage\experiments\artifacts\qwen_four_run_001\batch_summary.csv; report: C:\Users\moht2\Documents\Afterimage\experiments\artifacts\qwen_four_run_001\batch_report.md
[2026-11-10T10:18:29+00:00] r32_clahe_on / verify: exit 0; log=C:\Users\moht2\Documents\Afterimage\experiments\artifacts\qwen_four_run_001\r32_clahe_on\verify.log
[2026-11-10T10:18:29+00:00] Rechecking source, configuration, frozen CSV/image and cached-base hashes: r32_clahe_on
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
W1110 13:18:41.924000 11024 site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling regis

In [27]:
# Saved stage status, adapter settings and metrics — r32_clahe_on
orchestrator.show_run("r32_clahe_on")


Run: r32_clahe_on
Artifacts: C:\Users\moht2\Documents\Afterimage\experiments\artifacts\qwen_four_run_001\r32_clahe_on
Selected: True; eligible: True; exclusion: ; blockers: []

train: {
  "attempts": [
    {
      "started_at": "2026-11-10T09:24:55+00:00",
      "status": "complete",
      "ended_at": "2026-11-10T10:18:07+00:00",
      "elapsed_seconds": 3192.8899999999976
    }
  ],
  "status": "complete",
  "updated_at": "2026-11-10T10:18:07+00:00",
  "elapsed_seconds": 3192.8899999999976
}

verify: {
  "attempts": [
    {
      "started_at": "2026-11-10T10:18:12+00:00",
      "status": "complete",
      "ended_at": "2026-11-10T10:18:20+00:00",
      "elapsed_seconds": 7.077999999997701
    },
    {
      "started_at": "2026-11-10T10:18:21+00:00",
      "status": "complete",
      "ended_at": "2026-11-10T10:18:28+00:00",
      "elapsed_seconds": 7.0
    }
  ],
  "status": "complete",
  "updated_at": "2026-11-10T10:18:28+00:00",
  "elapsed_seconds": 7.0
}

evaluate: {
  "attempts": [


In [28]:
# Saved train log — r32_clahe_on; increase lines to inspect more history.
orchestrator.show_log("r32_clahe_on", "train", lines=40)


Log: C:\Users\moht2\Documents\Afterimage\experiments\artifacts\qwen_four_run_001\r32_clahe_on\train.log
 85%|████████▍ | 39/46 [00:27<00:05,  1.36it/s]

 87%|████████▋ | 40/46 [00:28<00:04,  1.36it/s]

 89%|████████▉ | 41/46 [00:29<00:03,  1.36it/s]

 91%|█████████▏| 42/46 [00:30<00:02,  1.36it/s]

 93%|█████████▎| 43/46 [00:30<00:02,  1.36it/s]

 96%|█████████▌| 44/46 [00:31<00:01,  1.36it/s]

 98%|█████████▊| 45/46 [00:32<00:00,  1.37it/s]

100%|██████████| 46/46 [00:32<00:00,  1.50it/s]
                                                      


                                               
{'eval_loss': '0.008546', 'eval_runtime': '33.58', 'eval_samples_per_second': '10.9', 'eval_steps_per_second': '1.37', 'epoch': '3'}

  6%|▌         | 1101/18350 [52:33<8:59:42,  1.88s/it]

100%|██████████| 46/46 [00:32<00:00,  1.50it/s]

                                               C:\Users\moht2\.conda\envs\afterimage_env\Lib\site-packages\peft\utils\save_and_load.py:313: UserWarning: Could no

In [29]:
# Saved verify log — r32_clahe_on; increase lines to inspect more history.
orchestrator.show_log("r32_clahe_on", "verify", lines=40)


Log: C:\Users\moht2\Documents\Afterimage\experiments\artifacts\qwen_four_run_001\r32_clahe_on\verify.log

=== 2026-11-10T10:18:12+00:00 ===
[2026-11-10T10:18:12+00:00] Rechecking source, configuration, frozen CSV/image and cached-base hashes: r32_clahe_on
[2026-11-10T10:18:20+00:00] Saved adapter verified after training process exit: r32_clahe_on
[2026-11-10T10:18:20+00:00] Batch summary: C:\Users\moht2\Documents\Afterimage\experiments\artifacts\qwen_four_run_001\batch_summary.csv; report: C:\Users\moht2\Documents\Afterimage\experiments\artifacts\qwen_four_run_001\batch_report.md

=== 2026-11-10T10:18:21+00:00 ===
[2026-11-10T10:18:21+00:00] Rechecking source, configuration, frozen CSV/image and cached-base hashes: r32_clahe_on
[2026-11-10T10:18:28+00:00] Saved adapter verified after training process exit: r32_clahe_on
[2026-11-10T10:18:29+00:00] Batch summary: C:\Users\moht2\Documents\Afterimage\experiments\artifacts\qwen_four_run_001\batch_summary.csv; report: C:\Users\moht2\Document

In [30]:
# Saved evaluate log — r32_clahe_on; increase lines to inspect more history.
orchestrator.show_log("r32_clahe_on", "evaluate", lines=40)


Log: C:\Users\moht2\Documents\Afterimage\experiments\artifacts\qwen_four_run_001\r32_clahe_on\evaluate.log
[2026-11-10T10:33:40+00:00] r32_clahe_on messidor_external: 690/1057
[2026-11-10T10:33:48+00:00] r32_clahe_on messidor_external: 700/1057
[2026-11-10T10:33:57+00:00] r32_clahe_on messidor_external: 710/1057
[2026-11-10T10:34:05+00:00] r32_clahe_on messidor_external: 720/1057
[2026-11-10T10:34:13+00:00] r32_clahe_on messidor_external: 730/1057
[2026-11-10T10:34:22+00:00] r32_clahe_on messidor_external: 740/1057
[2026-11-10T10:34:31+00:00] r32_clahe_on messidor_external: 750/1057
[2026-11-10T10:34:41+00:00] r32_clahe_on messidor_external: 760/1057
[2026-11-10T10:34:51+00:00] r32_clahe_on messidor_external: 770/1057
[2026-11-10T10:35:02+00:00] r32_clahe_on messidor_external: 780/1057
[2026-11-10T10:35:12+00:00] r32_clahe_on messidor_external: 790/1057
[2026-11-10T10:35:22+00:00] r32_clahe_on messidor_external: 800/1057
[2026-11-10T10:35:32+00:00] r32_clahe_on messidor_external: 810/1

## Run 4 — r16_clahe_off

Rank 16; CLAHE Off; alpha/r 1. CLAHE ablation against the shared rank-16 control; requires verified pairing.

Run Training, Verify adapter and Evaluation separately. Evaluation also rechecks the retained adapter in a CPU child before loading its model. Configuration/results/log cells only read saved files. In unattended mode the three action cells skip; the inspection cells remain available.


In [ ]:
# Saved resolved configuration — r16_clahe_off
orchestrator.show_configuration("r16_clahe_off")


In [ ]:
# Training — r16_clahe_off
if not UNATTENDED_QUEUE:
    orchestrator.run_stage("train", "r16_clahe_off")


In [ ]:
# Verify saved adapter — r16_clahe_off
if not UNATTENDED_QUEUE:
    orchestrator.run_stage("verify", "r16_clahe_off")


In [ ]:
# Evaluation on both frozen cohorts — r16_clahe_off
if not UNATTENDED_QUEUE:
    orchestrator.run_stage("evaluate", "r16_clahe_off")


In [ ]:
# Saved stage status, adapter settings and metrics — r16_clahe_off
orchestrator.show_run("r16_clahe_off")


In [ ]:
# Saved train log — r16_clahe_off; increase lines to inspect more history.
orchestrator.show_log("r16_clahe_off", "train", lines=40)


In [ ]:
# Saved verify log — r16_clahe_off; increase lines to inspect more history.
orchestrator.show_log("r16_clahe_off", "verify", lines=40)


In [ ]:
# Saved evaluate log — r16_clahe_off; increase lines to inspect more history.
orchestrator.show_log("r16_clahe_off", "evaluate", lines=40)


## Saved batch comparison

Read-only view of all four saved statuses and metrics. Retraining 2 remains the previously reported reference: 80.65% APTOS accuracy, 0.8915 QWK and 0.9362 macro ROC-AUC. These supplied figures stay separate from newly measured results.

Use the per-run cells above to pinpoint a failure. After interruption or a shared failure, rerun Preflight before another stage. Completed stages are verified and skipped; evaluation-only retries keep the trained adapter. The kernel can be shut down after the active action returns.


In [31]:
orchestrator.show_batch_summary()


r16_clahe_on: complete; APTOS acc/QWK/AUC=0.7138964577656676/0.8954684195322774/0.9318620982765713; Messidor acc/QWK/AUC=0.5383159886471145/0.6366909911620692/0.8067606235340958
r8_clahe_on: complete; APTOS acc/QWK/AUC=0.6975476839237057/0.6743088532606141/0.8931349588174795; Messidor acc/QWK/AUC=0.44181646168401134/0.13125992341458848/0.675019828052349
r32_clahe_on: complete; APTOS acc/QWK/AUC=0.7465940054495913/0.8840939730185815/0.9365515325621067; Messidor acc/QWK/AUC=0.46263008514664145/0.5470917222430295/0.7830023996384016
r16_clahe_off: excluded_by_configuration; APTOS acc/QWK/AUC=//; Messidor acc/QWK/AUC=//
CSV: C:\Users\moht2\Documents\Afterimage\experiments\artifacts\qwen_four_run_001\batch_summary.csv
Report: C:\Users\moht2\Documents\Afterimage\experiments\artifacts\qwen_four_run_001\batch_report.md
